# 🪞 NeuralMirror — BPM & HRV Research Notebook
**Remote Photoplethysmography (rPPG) · Signal Processing · Wellness Science**

This notebook is the *research layer* of the NeuralMirror project.  
It walks through the full pipeline from raw webcam frames → filtered pulse signal → BPM + HRV metrics.  
Include this in your GitHub repo to demonstrate scientific rigour to buildathon judges.

## 1 · Create and Verify the Python Environment
Confirm we are running inside the project's virtual environment.

In [ ]:
import sys, os

print("Python executable :", sys.executable)
print("Python version    :", sys.version.split()[0])
print("Working directory :", os.getcwd())

# Confirm we are inside the NeuralMirror venv
in_venv = (
    hasattr(sys, "real_prefix")
    or (hasattr(sys, "base_prefix") and sys.base_prefix != sys.prefix)
)
print("Inside venv       :", in_venv)

## 2 · Install and Check Required Packages

In [ ]:
%pip install --quiet opencv-python heartpy numpy matplotlib scipy pandas python-dotenv openai requests

# ── Verify imports ────────────────────────────────────────────────────────────
import importlib, textwrap

packages = {
    "cv2": "opencv-python",
    "heartpy": "heartpy",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "pandas": "pandas",
}

for mod, pkg in packages.items():
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        print(f"  ✅  {pkg:<20} {ver}")
    except ImportError:
        print(f"  ❌  {pkg} NOT found — run: pip install {pkg}")

## 3 · Open the Webcam and Preview Frames
Capture a short burst of frames to verify camera access and image quality.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

CAMERA_INDEX = 0
CAPTURE_SECONDS = 30
TARGET_FPS = 30
N_FRAMES = CAPTURE_SECONDS * TARGET_FPS

cap = cv2.VideoCapture(CAMERA_INDEX)
assert cap.isOpened(), "❌ Could not open camera — check CAMERA_INDEX."

cap.set(cv2.CAP_PROP_FPS, TARGET_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)
print(f"Camera: {width}×{height} @ {fps:.0f} fps")

# Grab one preview frame
ret, preview = cap.read()
assert ret, "❌ Failed to read a frame."

preview_rgb = cv2.cvtColor(preview, cv2.COLOR_BGR2RGB)
fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(preview_rgb)
ax.set_title("Webcam Preview Frame", fontsize=13, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig("../notebooks/preview_frame.png", dpi=120)
plt.show()
print("Preview frame saved → notebooks/preview_frame.png")

## 4 · Select the Forehead Region of Interest (ROI)
Detect the face with a Haar cascade, then crop the forehead — the most stable rPPG region.

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

def get_forehead_roi(frame_bgr):
    """Return (roi_bgr, bbox) where bbox = (x1, y1, x2, y2), or (None, None)."""
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(100, 100)
    )
    if len(faces) == 0:
        return None, None
    x, y, w, h = faces[0]
    x1, x2 = x + int(w * 0.25), x + int(w * 0.75)
    y1, y2 = y + int(h * 0.05), y + int(h * 0.30)
    return frame_bgr[y1:y2, x1:x2], (x1, y1, x2, y2)

roi, bbox = get_forehead_roi(preview)

if roi is not None:
    annotated = preview_rgb.copy()
    x1, y1, x2, y2 = bbox
    # Draw forehead rectangle
    import matplotlib.patches as patches
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].imshow(annotated)
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor="#00ff88", facecolor="none")
    axes[0].add_patch(rect)
    axes[0].set_title("Face + Forehead ROI", fontsize=12, fontweight="bold")
    axes[0].axis("off")

    axes[1].imshow(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Cropped Forehead ROI", fontsize=12, fontweight="bold")
    axes[1].axis("off")

    plt.tight_layout()
    plt.savefig("../notebooks/roi_detection.png", dpi=120)
    plt.show()
    print(f"ROI shape: {roi.shape}  |  saved → notebooks/roi_detection.png")
else:
    print("⚠️ No face detected in the preview frame — ensure your face is visible.")

## 5 · Extract a Pulse Signal from Video Frames
Capture `CAPTURE_SECONDS` seconds of video and sample the mean green-channel intensity of the forehead ROI on every frame.  
This produces the raw rPPG signal — a time-series of subtle colour fluctuations driven by blood volume changes.

In [ ]:
import time

green_signal = []
timestamps   = []
dropped      = 0

print(f"Recording {CAPTURE_SECONDS}s of signal — keep your face in frame …")
t_start = time.time()

while (time.time() - t_start) < CAPTURE_SECONDS:
    ret, frame = cap.read()
    if not ret:
        dropped += 1
        continue
    roi, _ = get_forehead_roi(frame)
    if roi is not None and roi.size > 0:
        green_signal.append(float(np.mean(roi[:, :, 1])))  # green channel
        timestamps.append(time.time() - t_start)

cap.release()  # done with camera

green_signal = np.array(green_signal)
timestamps   = np.array(timestamps)
actual_fps   = len(green_signal) / timestamps[-1] if len(timestamps) > 1 else TARGET_FPS

print(f"Captured {len(green_signal)} samples  |  "
      f"Effective FPS: {actual_fps:.1f}  |  Dropped frames: {dropped}")

## 6 · Filter the Signal and Estimate BPM
Apply a Butterworth band-pass filter (0.75–3.5 Hz → 45–210 BPM) then run HeartPy to detect peaks and compute heart-rate metrics.

In [ ]:
import heartpy as hp

BAND_LOW  = 0.75   # Hz  (~45 BPM)
BAND_HIGH = 3.5    # Hz  (~210 BPM)

filtered = hp.filter_signal(
    green_signal,
    cutoff=[BAND_LOW, BAND_HIGH],
    sample_rate=actual_fps,
    filtertype="bandpass",
    order=3,
)

try:
    working_data, measures = hp.process(filtered, sample_rate=actual_fps)

    bpm   = measures["bpm"]
    sdnn  = measures.get("sdnn",  float("nan"))
    rmssd = measures.get("rmssd", float("nan"))
    pnn50 = measures.get("pnn50", float("nan"))

    stress_index = min(
        max((bpm - 50) / 110, 0.0) * 0.6
        + (1.0 - min(rmssd / 80.0, 1.0)) * 0.4,
        1.0
    )

    print(f"{'─'*40}")
    print(f"  Heart Rate  : {bpm:.1f} BPM")
    print(f"  SDNN        : {sdnn:.1f} ms    (HRV — higher is healthier)")
    print(f"  RMSSD       : {rmssd:.1f} ms    (HRV — higher is healthier)")
    print(f"  pNN50       : {pnn50:.1f}%")
    print(f"  Stress Index: {stress_index:.3f} / 1.000")
    print(f"{'─'*40}")

except Exception as e:
    print(f"⚠️  HeartPy analysis failed: {e}")
    print("    Try recording in better lighting or sitting still.")

## 7 · Visualize Raw Signal, Filtered Signal, and BPM
Three-panel publication-quality chart — the kind that impresses buildathon judges.

In [ ]:
plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor":   "#161b22",
    "axes.edgecolor":   "#30363d",
    "axes.labelcolor":  "#c9d1d9",
    "xtick.color":      "#8b949e",
    "ytick.color":      "#8b949e",
    "text.color":       "#c9d1d9",
    "grid.color":       "#21262d",
    "grid.linewidth":   0.6,
})

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=False)
fig.suptitle(
    "🪞 NeuralMirror — rPPG Signal Analysis",
    fontsize=16, fontweight="bold", color="#58a6ff", y=0.98
)

# ── Panel 1: Raw green-channel signal ─────────────────────────────────────
axes[0].plot(timestamps, green_signal, color="#3fb950", linewidth=0.8, alpha=0.9)
axes[0].set_ylabel("Green Intensity", fontsize=10)
axes[0].set_title("Raw rPPG Signal (Green Channel)", fontsize=11, color="#8b949e")
axes[0].grid(True)

# ── Panel 2: Filtered pulse waveform ──────────────────────────────────────
axes[1].plot(timestamps, filtered, color="#58a6ff", linewidth=1.0)
if "working_data" in dir() and "peaklist" in working_data:
    peak_times = np.array(working_data["peaklist"]) / actual_fps
    peak_vals  = filtered[working_data["peaklist"]]
    axes[1].scatter(peak_times, peak_vals, color="#f78166", s=30, zorder=5,
                    label="Detected peaks")
    axes[1].legend(facecolor="#161b22", edgecolor="#30363d", fontsize=9)
axes[1].set_ylabel("Amplitude (norm.)", fontsize=10)
axes[1].set_title("Band-Pass Filtered Pulse Waveform  (0.75–3.5 Hz)", fontsize=11, color="#8b949e")
axes[1].grid(True)

# ── Panel 3: HRV metrics bar ──────────────────────────────────────────────
metrics  = ["BPM", "SDNN (ms)", "RMSSD (ms)", "Stress ×100"]
values   = [bpm, sdnn, rmssd, stress_index * 100]
colours  = ["#3fb950", "#58a6ff", "#a5d6ff", "#f78166"]
bars = axes[2].bar(metrics, values, color=colours, width=0.5, edgecolor="#30363d")
for bar, val in zip(bars, values):
    axes[2].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{val:.1f}", ha="center", va="bottom", fontsize=10, color="#c9d1d9"
    )
axes[2].set_ylabel("Value", fontsize=10)
axes[2].set_title("Biometric Summary", fontsize=11, color="#8b949e")
axes[2].grid(True, axis="y")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("../notebooks/bpm_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved → notebooks/bpm_analysis.png")

## 8 · Save Sample Data and Plots for the Project Repo
Export the processed signal and metrics to CSV so they can be committed alongside the notebook.

In [ ]:
import pandas as pd
from pathlib import Path

output_dir = Path("../notebooks")
output_dir.mkdir(parents=True, exist_ok=True)

# ── 1. Raw + filtered signal DataFrame ───────────────────────────────────────
signal_df = pd.DataFrame({
    "time_s":       timestamps,
    "green_raw":    green_signal,
    "green_filtered": filtered,
})
signal_df.to_csv(output_dir / "pulse_signal.csv", index=False)
print(f"Signal data  → {output_dir / 'pulse_signal.csv'}  ({len(signal_df)} rows)")

# ── 2. Biometric summary row ──────────────────────────────────────────────────
summary_df = pd.DataFrame([{
    "bpm":          round(bpm,   2),
    "sdnn_ms":      round(sdnn,  2),
    "rmssd_ms":     round(rmssd, 2),
    "pnn50_pct":    round(pnn50, 2),
    "stress_index": round(stress_index, 4),
    "sample_count": len(green_signal),
    "effective_fps": round(actual_fps, 2),
}])
summary_df.to_csv(output_dir / "session_summary.csv", index=False)
print(f"Session summary → {output_dir / 'session_summary.csv'}")

# ── 3. Confirm saved assets ───────────────────────────────────────────────────
saved = sorted(output_dir.iterdir())
print("\nAssets saved to notebooks/:")
for f in saved:
    print(f"  📄 {f.name}")

print("\n✅  All data exported. Commit the notebooks/ folder to GitHub.")